<a href="https://colab.research.google.com/github/mokshmahajan2004/AI-Lab-Assignments/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab Assignment-4 (Question 1)

Develop an AI agent that plays Tic-Tac-Toe optimally and optimizes it further using alpha-beta pruning.

Assignment
Tasks: Game setup -> Min-Max Algorithm -> Alpha-Beta Pruning -> Performance comparison (with and without pruning)


# **Solution:**

In [ ]:
import math
import copy

class TicTacToe:
    def __init__(self):
        self.board = [" " for _ in range(9)]  # 3x3 board
        self.current_winner = None

    def print_board(self):
        for row in [self.board[i*3:(i+1)*3] for i in range(3)]:
            print("| " + " | ".join(row) + " |")

    def available_moves(self):
        return [i for i, spot in enumerate(self.board) if spot == " "]

    def make_move(self, square, letter):
        if self.board[square] == " ":
            self.board[square] = letter
            if self.winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def winner(self, square, letter):
        row_ind = square // 3
        row = self.board[row_ind*3:(row_ind+1)*3]
        if all([spot == letter for spot in row]):
            return True
        col_ind = square % 3
        col = [self.board[col_ind+i*3] for i in range(3)]
        if all([spot == letter for spot in col]):
            return True
        # diagonals
        if square % 2 == 0:
            diag1 = [self.board[i] for i in [0, 4, 8]]
            diag2 = [self.board[i] for i in [2, 4, 6]]
            if all([spot == letter for spot in diag1]) or all([spot == letter for spot in diag2]):
                return True
        return False

    def empty_squares(self):
        return " " in self.board

    def num_empty_squares(self):
        return len(self.available_moves())


# **Minimax Implementation**

In [ ]:
def minimax(state, player, maximizing_player, nodes_counter):
    nodes_counter[0] += 1
    opponent = "O" if player == "X" else "X"

    if state.current_winner == opponent:
        return {"position": None, "score": 1 * (state.num_empty_squares() + 1) if opponent == maximizing_player else -1 * (state.num_empty_squares() + 1)}

    elif not state.empty_squares():
        return {"position": None, "score": 0}

    if player == maximizing_player:
        best = {"position": None, "score": -math.inf}
    else:
        best = {"position": None, "score": math.inf}

    for move in state.available_moves():
        new_state = copy.deepcopy(state)
        new_state.make_move(move, player)
        sim_score = minimax(new_state, "O" if player == "X" else "X", maximizing_player, nodes_counter)

        sim_score["position"] = move
        if player == maximizing_player:
            if sim_score["score"] > best["score"]:
                best = sim_score
        else:
            if sim_score["score"] < best["score"]:
                best = sim_score
    return best


# **Alpha-Beta Pruning Implementation**

In [ ]:
def alpha_beta(state, player, maximizing_player, alpha, beta, nodes_counter):
    nodes_counter[0] += 1
    opponent = "O" if player == "X" else "X"

    if state.current_winner == opponent:
        return {"position": None, "score": 1 * (state.num_empty_squares() + 1) if opponent == maximizing_player else -1 * (state.num_empty_squares() + 1)}

    elif not state.empty_squares():
        return {"position": None, "score": 0}

    if player == maximizing_player:
        best = {"position": None, "score": -math.inf}
        for move in state.available_moves():
            new_state = copy.deepcopy(state)
            new_state.make_move(move, player)
            sim_score = alpha_beta(new_state, "O" if player == "X" else "X", maximizing_player, alpha, beta, nodes_counter)
            sim_score["position"] = move
            if sim_score["score"] > best["score"]:
                best = sim_score
            alpha = max(alpha, sim_score["score"])
            if beta <= alpha:
                break
    else:
        best = {"position": None, "score": math.inf}
        for move in state.available_moves():
            new_state = copy.deepcopy(state)
            new_state.make_move(move, player)
            sim_score = alpha_beta(new_state, "O" if player == "X" else "X", maximizing_player, alpha, beta, nodes_counter)
            sim_score["position"] = move
            if sim_score["score"] < best["score"]:
                best = sim_score
            beta = min(beta, sim_score["score"])
            if beta <= alpha:
                break
    return best


# **Performance Comparison Example**

In [ ]:
game = TicTacToe()

# Minimax
nodes_minimax = [0]
best_move = minimax(game, "X", "X", nodes_minimax)
print("Minimax Best Move:", best_move)
print("Nodes explored (Minimax):", nodes_minimax[0])

# Alpha-Beta
game = TicTacToe()  # Reset game
nodes_ab = [0]
best_move_ab = alpha_beta(game, "X", "X", -math.inf, math.inf, nodes_ab)
print("Alpha-Beta Best Move:", best_move_ab)
print("Nodes explored (Alpha-Beta):", nodes_ab[0])


Minimax Best Move: {'position': 0, 'score': 0}
Nodes explored (Minimax): 549946
Alpha-Beta Best Move: {'position': 0, 'score': 0}
Nodes explored (Alpha-Beta): 20866


# **Lab Assignment-4 (Question 2)**

Model and solve a warehouse robot navigation task using Markov Decision Process (MDP),
and find the optimal policy for the robot to reach the target while maximizing
rewards and avoiding obstacles.

A robot operates in a 4x4 grid-based warehouse. Each cell in the grid represents a state,
and the robot can perform one of the following actions at each state:
UP, DOWN, LEFT, RIGHT
The robot aims to navigate from a start state to a goal state while maximizing cumulative rewards.

Grid Layout Example:


S  0   -1  G

0 -10  0  0

0   0   0 -10

0  -1   0   0

Here
·     S: Start state
·     G: Goal state (reward = +10)
·     -1: Light obstacle (small penalty)
·     -10:Heavy obstacle (large penalty)
·      0: Normal cells (no reward)

MDP Components:
States (S): Each cell in the grid (16 total)
Actions (A): {UP, DOWN, LEFT, RIGHT}
Transition Probabilities (P):
The robot succeeds in moving in the intended direction with probability
0.8, and with 0.1 it moves in a perpendicular direction (left or right of
intended direction). If it hits a wall, it stays in place.
Rewards (R):
+10 for reaching the goal
-10 for heavy obstacles
-1 for light obstacles
0 for empty cells
Discount Factor (γ): 0.9

Tasks for Students:
Define the MDP:

Formally define the states, actions, transition probabilities, rewards, and discount factors.

-> Implement MDP Solution Algorithms:
Value Iteration
-> Compute and Display:
Optimal value function (as a grid)
Optimal policy (using arrows: ↑ ↓ → ←)

➤ States (S):
Each cell in the 4x4 grid is a state: total of 16 states, indexed as (i, j) from top-left (0,0) to bottom-right (3,3).

➤ Actions (A):
["UP", "DOWN", "LEFT", "RIGHT"]

➤ Transition Probabilities (P):
Intended direction: 0.8

Perpendicular directions (left/right of intended): 0.1 each

If a move goes outside the grid → stays in place.

➤ Rewards (R):

Symbol	Description	Reward
S	Start	0
G	Goal	+10
-1	Light obstacle	-1
-10	Heavy obstacle	-10
0	Empty cell	0

➤ Discount Factor (γ):
gamma = 0.9



# **Grid Setup**

In [ ]:
import numpy as np

# Grid definition (based on the question)
grid = [
    ['S',  0,   -1, 'G'],
    [ 0 , -10,   0,  0],
    [ 0 ,  0,    0, -10],
    [ 0 , -1,    0,  0]
]

rewards = {
    'S': 0,
    'G': 10,
    -1: -1,
    -10: -10,
    0: 0
}

actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
action_vectors = {
    'UP': (-1, 0),
    'DOWN': (1, 0),
    'LEFT': (0, -1),
    'RIGHT': (0, 1)
}


# **3. Helper Functions**

In [ ]:
def in_bounds(i, j):
    return 0 <= i < 4 and 0 <= j < 4

def get_reward(i, j):
    cell = grid[i][j]
    return rewards[cell]

def is_terminal(i, j):
    return grid[i][j] == 'G'


# **4. Transition Function**

In [ ]:
def get_transitions(i, j, action):
    directions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
    perp = {
        'UP': ['LEFT', 'RIGHT'],
        'DOWN': ['LEFT', 'RIGHT'],
        'LEFT': ['UP', 'DOWN'],
        'RIGHT': ['UP', 'DOWN']
    }

    transitions = []
    for a, prob in [(action, 0.8)] + [(d, 0.1) for d in perp[action]]:
        di, dj = action_vectors[a]
        ni, nj = i + di, j + dj
        if not in_bounds(ni, nj):
            ni, nj = i, j
        transitions.append((prob, ni, nj))
    return transitions


# **5. Value Iteration**

In [ ]:
def value_iteration(gamma=0.9, theta=1e-4):
    V = np.zeros((4, 4))
    policy = [['' for _ in range(4)] for _ in range(4)]

    while True:
        delta = 0
        new_V = V.copy()

        for i in range(4):
            for j in range(4):
                if is_terminal(i, j):
                    continue

                max_value = float('-inf')
                best_action = None

                for action in actions:
                    total = 0
                    for prob, ni, nj in get_transitions(i, j, action):
                        r = get_reward(ni, nj)
                        total += prob * (r + gamma * V[ni][nj])

                    if total > max_value:
                        max_value = total
                        best_action = action

                new_V[i][j] = max_value
                policy[i][j] = best_action
                delta = max(delta, abs(V[i][j] - new_V[i][j]))

        V = new_V
        if delta < theta:
            break

    return V, policy


# **6. Display Results**

In [ ]:
def print_value_grid(V):
    print("Optimal Value Function (rounded):")
    for row in V:
        print(["{0:>6.2f}".format(v) for v in row])
    print()

def print_policy(policy):
    symbols = {'UP': '↑', 'DOWN': '↓', 'LEFT': '←', 'RIGHT': '→', None: '.'}
    print("Optimal Policy:")
    for i in range(4):
        row = []
        for j in range(4):
            if is_terminal(i, j):
                row.append(' G ')
            else:
                act = policy[i][j]
                row.append(f' {symbols.get(act, ".")} ')
        print(' '.join(row))


# **7. Run It**

In [ ]:
V, optimal_policy = value_iteration()
print_value_grid(V)
print_policy(optimal_policy)


Optimal Value Function (rounded):
['  5.28', '  6.21', '  9.49', '  0.00']
['  3.76', '  6.84', '  8.20', '  9.60']
['  3.61', '  4.15', '  5.91', '  7.08']
['  3.15', '  4.13', '  4.84', '  3.43']

Optimal Policy:
 →   →   →   G 
 ↑   →   →   ↑ 
 →   →   ↑   ↑ 
 ↑   →   ↑   ← 


# **Lab Assignment-4 (Question 3)**

Implement the policy iteration algorithm to solve the MDP in Question 2 of Assignment 4.

In [ ]:
import numpy as np

# Grid and reward setup
grid = [
    ['S',  0,   -1, 'G'],
    [ 0 , -10,   0,  0],
    [ 0 ,  0,    0, -10],
    [ 0 , -1,    0,  0]
]

rewards = {
    'S': 0,
    'G': 10,
    -1: -1,
    -10: -10,
    0: 0
}

actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
action_vectors = {
    'UP': (-1, 0),
    'DOWN': (1, 0),
    'LEFT': (0, -1),
    'RIGHT': (0, 1)
}

def in_bounds(i, j):
    return 0 <= i < 4 and 0 <= j < 4

def get_reward(i, j):
    cell = grid[i][j]
    return rewards[cell]

def is_terminal(i, j):
    return grid[i][j] == 'G'

def get_transitions(i, j, action):
    perp = {
        'UP': ['LEFT', 'RIGHT'],
        'DOWN': ['LEFT', 'RIGHT'],
        'LEFT': ['UP', 'DOWN'],
        'RIGHT': ['UP', 'DOWN']
    }

    transitions = []
    for a, prob in [(action, 0.8)] + [(d, 0.1) for d in perp[action]]:
        di, dj = action_vectors[a]
        ni, nj = i + di, j + dj
        if not in_bounds(ni, nj):
            ni, nj = i, j
        transitions.append((prob, ni, nj))
    return transitions

def policy_iteration(gamma=0.9, theta=1e-4):
    V = np.zeros((4, 4))
    policy = [[np.random.choice(actions) for _ in range(4)] for _ in range(4)]

    while True:
        # Policy Evaluation
        while True:
            delta = 0
            new_V = V.copy()
            for i in range(4):
                for j in range(4):
                    if is_terminal(i, j):
                        continue
                    action = policy[i][j]
                    value = 0
                    for prob, ni, nj in get_transitions(i, j, action):
                        r = get_reward(ni, nj)
                        value += prob * (r + gamma * V[ni][nj])
                    new_V[i][j] = value
                    delta = max(delta, abs(V[i][j] - new_V[i][j]))
            V = new_V
            if delta < theta:
                break

        # Policy Improvement
        policy_stable = True
        for i in range(4):
            for j in range(4):
                if is_terminal(i, j):
                    continue
                old_action = policy[i][j]
                best_action = None
                best_value = float('-inf')
                for action in actions:
                    value = 0
                    for prob, ni, nj in get_transitions(i, j, action):
                        r = get_reward(ni, nj)
                        value += prob * (r + gamma * V[ni][nj])
                    if value > best_value:
                        best_value = value
                        best_action = action
                policy[i][j] = best_action
                if old_action != best_action:
                    policy_stable = False
        if policy_stable:
            break

    return V, policy

def print_value_grid(V):
    print("Optimal Value Function:")
    for row in V:
        print(["{0:>6.2f}".format(v) for v in row])
    print()

def print_policy(policy):
    symbols = {'UP': '↑', 'DOWN': '↓', 'LEFT': '←', 'RIGHT': '→', None: '.'}
    print("Optimal Policy:")
    for i in range(4):
        row = []
        for j in range(4):
            if is_terminal(i, j):
                row.append(' G ')
            else:
                row.append(f" {symbols.get(policy[i][j], '.')} ")
        print(' '.join(row))

# Run policy iteration
V, optimal_policy = policy_iteration()
print_value_grid(V)
print_policy(optimal_policy)


Optimal Value Function:
['  5.28', '  6.21', '  9.49', '  0.00']
['  3.76', '  6.84', '  8.20', '  9.60']
['  3.61', '  4.15', '  5.91', '  7.08']
['  3.15', '  4.13', '  4.84', '  3.43']

Optimal Policy:
 →   →   →   G 
 ↑   →   →   ↑ 
 →   →   ↑   ↑ 
 ↑   →   ↑   ← 


Lab Assignment-4 (Question 4)

An agent navigates a 5x5 grid world. It can take actions: UP, DOWN, LEFT, RIGHT.
The goal is to reach a terminal state that provides a positive reward while
avoiding traps that give negative rewards.

In [ ]:
import numpy as np
from collections import defaultdict

# -----------------------------
# STEP 1: Define training data
# -----------------------------
training_episodes = [
    [((0, 0),'U', 0, (0, 0)), ((0, 0),'D', 0, (1, 0)), ((0, 0),'L', 0, (0, 0)), ((0, 0),'R', -1, (0, 1))],
    [((0, 1),'U', 0, (0, 1)), ((0, 1),'D', 0, (1, 1)), ((0, 1),'L', 0, (0, 0)), ((0, 1),'R', 0, (0, 2))],
    [((0, 2),'U', 0, (0, 2)), ((0, 2),'D', 0, (1, 2)), ((0, 2),'L', -1, (0, 1)), ((0, 2),'R', 0, (0, 3))],
    [((0, 3),'U', 0, (0, 3)), ((0, 3),'D', 0, (1, 3)), ((0, 3),'L', 0, (0, 2)), ((0, 3),'R', 0, (0, 4))],
    [((0, 4),'U', 0, (0, 4)), ((0, 4),'D', 0, (1, 4)), ((0, 4),'L', 0, (0, 3)), ((0, 4),'R', 0, (0, 4))],
    [((1, 0),'U', 0, (0, 0)), ((1, 0),'D', 0, (2, 0)), ((1, 0),'L', 0, (1, 0)), ((1, 0),'R', 0, (1, 1))],
    [((1, 1),'U', -1, (0, 1)), ((1, 1),'D', -10, (2, 1)), ((1, 1),'L', 0, (1, 0)), ((1, 1),'R', 0, (1, 2))],
    [((1, 2),'U', 0, (0, 2)), ((1, 2),'D', 0, (2, 2)), ((1, 2),'L', 0, (1, 1)), ((1, 2),'R', 0, (1, 3))],
    [((1, 3),'U', 0, (0, 3)), ((1, 3),'D', 0, (2, 3)), ((1, 3),'L', 0, (1, 2)), ((1, 3),'R', 0, (1, 4))],
    [((1, 4),'U', 0, (0, 4)), ((1, 4),'D', 0, (2, 4)), ((1, 4),'L', 0, (1, 3)), ((1, 4),'R', 0, (1, 4))],
    [((2, 0),'U', 0, (1, 0)), ((2, 0),'D', 0, (3, 0)), ((2, 0),'L', 0, (2, 0)), ((2, 0),'R', -10, (2, 1))],
    [((2, 1),'U', 0, (1, 1)), ((2, 1),'D', 0, (3, 1)), ((2, 1),'L', 0, (2, 0)), ((2, 1),'R', 0, (2, 2))],
    [((2, 2),'U', 0, (1, 2)), ((2, 2),'D', 0, (3, 2)), ((2, 2),'L', -10, (2, 1)), ((2, 2),'R', 0, (2, 3))],
    [((2, 3),'U', 0, (1, 3)), ((2, 3),'D', 0, (3, 3)), ((2, 3),'L', 0, (2, 2)), ((2, 3),'R', 0, (2, 4))],
    [((2, 4),'U', 0, (1, 4)), ((2, 4),'D', 10, (3, 4)), ((2, 4),'L', 0, (2, 3)), ((2, 4),'R', 0, (2, 4))],
    [((3, 0),'U', 0, (2, 0)), ((3, 0),'D', 0, (4, 0)), ((3, 0),'L', 0, (3, 0)), ((3, 0),'R', 0, (3, 1))],
    [((3, 1),'U', -10, (2, 1)), ((3, 1),'D', 0, (4, 1)), ((3, 1),'L', 0, (3, 0)), ((3, 1),'R', 0, (3, 2))],
    [((3, 2),'U', 0, (2, 2)), ((3, 2),'D', 0, (4, 2)), ((3, 2),'L', 0, (3, 1)), ((3, 2),'R', 0, (3, 3))],
    [((3, 3),'U', 0, (2, 3)), ((3, 3),'D', 0, (4, 3)), ((3, 3),'L', 0, (3, 2)), ((3, 3), 'R', 10, (3, 4))],
    [((3, 4),'U', 0, (2, 4)), ((3, 4),'D', 0, (4, 4)), ((3, 4),'L', 0, (3, 3)), ((3, 4),'R', 0, (3, 4))],
    [((4, 0),'U', 0, (3, 0)), ((4, 0),'D', 0, (4, 0)), ((4, 0),'L', 0, (4, 0)), ((4, 0),'R', 0, (4, 1))],
    [((4, 1),'U', 0, (3, 1)), ((4, 1),'D', 0, (4, 1)), ((4, 1),'L', 0, (4, 0)), ((4, 1),'R', 0, (4, 2))],
    [((4, 2),'U', 0, (3, 2)), ((4, 2),'D', 0, (4, 2)), ((4, 2),'L', 0, (4, 1)), ((4, 2),'R', 0, (4, 3))],
    [((4, 3),'U', 0, (3, 3)), ((4, 3),'D', 0, (4, 3)), ((4, 3),'L', 0, (4, 2)), ((4, 3),'R', 0, (4, 4))],
    [((4, 4),'U', 10, (3, 4)), ((4, 4),'D', 0, (4, 4)), ((4, 4),'L', 0, (4, 3)), ((4, 4),'R', 0, (4, 4))]
]

# -----------------------------
# STEP 2: Learn the model
# -----------------------------
T = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))  # T[s][a][s']
R = defaultdict(lambda: defaultdict(list))  # R[s][a] = list of rewards

for episode in training_episodes:
    for (s, a, r, s_) in episode:
        T[s][a][s_] += 1
        R[s][a].append(r)

# Normalize transitions and average rewards
P = {}
R_avg = {}
for s in T:
    P[s] = {}
    R_avg[s] = {}
    for a in T[s]:
        total = sum(T[s][a].values())
        P[s][a] = {s2: count / total for s2, count in T[s][a].items()}
        R_avg[s][a] = sum(R[s][a]) / len(R[s][a])

# -----------------------------
# STEP 3: Value Iteration
# -----------------------------
gamma = 0.9
theta = 1e-4
V = defaultdict(float)
states = [(i, j) for i in range(5) for j in range(5)]
actions = ['U', 'D', 'L', 'R']

while True:
    delta = 0
    for s in states:
        if s not in P:
            continue
        max_v = float('-inf')
        for a in P[s]:
            v = sum(P[s][a][s_]*(R_avg[s][a] + gamma * V[s_]) for s_ in P[s][a])
            max_v = max(max_v, v)
        delta = max(delta, abs(V[s] - max_v))
        V[s] = max_v
    if delta < theta:
        break

# -----------------------------
# STEP 4: Extract Optimal Policy
# -----------------------------
policy = {}
for s in states:
    best_a, best_v = None, float('-inf')
    for a in P.get(s, {}):
        v = sum(P[s][a][s_]*(R_avg[s][a] + gamma * V[s_]) for s_ in P[s][a])
        if v > best_v:
            best_v = v
            best_a = a
    policy[s] = best_a

# -----------------------------
# STEP 5: Display results
# -----------------------------
def print_policy(policy):
    grid = []
    arrows = {'U': '↑', 'D': '↓', 'L': '←', 'R': '→', None: '.'}
    for i in range(5):
        row = []
        for j in range(5):
            row.append(arrows.get(policy.get((i, j), None), '.'))
        grid.append(row)
    print("Optimal Policy:")
    for row in grid:
        print(' '.join(row))

def print_values(V):
    print("Value Function:")
    for i in range(5):
        row = []
        for j in range(5):
            row.append("{:6.2f}".format(V.get((i, j), 0)))
        print(' '.join(row))

print_values(V)
print_policy(policy)


Value Function:
 27.97  31.08  34.53  38.37  42.63
 31.08  34.53  38.37  42.63  47.37
 34.53  38.37  42.63  47.37  52.63
 38.37  42.63  47.37  52.63  47.37
 34.53  38.37  42.63  47.37  52.63
Optimal Policy:
↓ ↓ ↓ ↓ ↓
↓ → ↓ ↓ ↓
↓ ↓ ↓ ↓ ↓
→ → → → ↓
→ → → → ↑
